# Danbooru Provider Probe

This notebook validates the temporary Booru abstraction under `tmp/lib/booru` using Danbooru only. Danbooru public search does not require an API key.

Test scope:
- Search posts
- Inspect normalized `BooruPost` fields
- Fetch a single post by ID
- Fetch tag suggestions
- Check thumbnail URL availability with `HEAD`

In [ ]:
from pprint import pprint

import requests
from requests import HTTPError

from tmp.lib.booru import DanbooruClient, create_booru_client


def print_http_error(error):
    response = getattr(error, "response", None)
    print(type(error).__name__, error)
    if response is not None:
        print("status:", response.status_code)
        print("content-type:", response.headers.get("Content-Type"))
        print(response.text[:500].replace("\n", " "))


In [ ]:
client = create_booru_client("danbooru")
client


## Search Posts

In [ ]:
result = None
try:
    result = client.search_posts("cat rating:general", limit=3)
    print(result.provider, result.tags, result.page, result.limit, result.has_next)
    print("posts:", len(result.posts))

    for post in result.posts:
        print({
            "id": post.id,
            "page_url": post.page_url,
            "thumb": post.thumbnail_url,
            "sample": post.sample_url,
            "original": post.original_url,
            "size": (post.width, post.height),
            "rating": post.rating,
            "score": post.score,
            "tag_count": len(post.flat_tags),
        })
except HTTPError as error:
    print_http_error(error)


## Fetch Single Post

In [ ]:
if result and result.posts:
    first = result.posts[0]
    single = client.get_post(first.id)

    print(single.id, single.page_url)
    print(single.thumbnail_url)
    print(single.original_url)
    print(single.tags.keys())
else:
    print("Search did not return posts; skipping single-post fetch.")


## Tag Suggestions

In [ ]:
try:
    suggestions = client.tag_suggestions("cat", limit=5)
    for suggestion in suggestions:
        print(suggestion.tag, suggestion.type, suggestion.count)
except HTTPError as error:
    print_http_error(error)


## Thumbnail Availability

In [ ]:
if result and result.posts:
    thumb_url = result.posts[0].thumbnail_url
    response = client.session.head(thumb_url, timeout=20, allow_redirects=True)
    print(response.status_code)
    print(response.headers.get("Content-Type"))
    print(response.headers.get("Content-Length"))
else:
    print("Search did not return posts; skipping thumbnail HEAD check.")


## Pagination Smoke Check

In [ ]:
if result and result.posts:
    next_page = client.next_page(result)
    next_result = client.search_posts(result.tags, page=next_page, limit=3)
    print(next_result.page, len(next_result.posts))
    print([post.id for post in next_result.posts])
else:
    print("Search did not return posts; skipping pagination check.")
